# 02 Recursive Chunking

This notebook is part of the **Production-Style PDF Chatbot / RAG System Project**.

## Objectives
- Understand the underlying concepts.
- Build a production-quality implementation.
- Learn the theory behind every code block.


In [2]:
# # Start coding here
# !pip install langchain-text-splitters
# !pip install pandas

In [3]:
# !pip install langchain-text-splitters
# !pip install pandas

In [4]:
# ==========================================================
# Notebook 02: Recursive Chunking
# ==========================================================

import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
pages_df = pd.read_csv("data/processed/parsed_documents.csv")

pages_df["source_document"].unique()

array(['annual_report.pdf'], dtype=object)

In [7]:
pages_df = pd.read_csv("data/processed/parsed_documents.csv")

pages_df.head()

,page_number,page_text,clean_text,source_document
0,1,ABBOTT INDIA LIMITED\nANNUAL REPORT 2024-25\nT...,ABBOTT INDIA LIMITED ANNUAL REPORT 2024-25 TOG...,annual_report.pdf
1,2,Abbott India Limited\nCONTENTS\nCompany Overvi...,Abbott India Limited CONTENTS Company Overview...,annual_report.pdf
2,3,CCoommppaannyy OOvveerrvviieeww SSttaattuuttoo...,CCoommppaannyy OOvveerrvviieeww SSttaattuuttoo...,annual_report.pdf
3,4,CCoommppaannyy OOvveerrvviieeww SSttaattuuttoo...,CCoommppaannyy OOvveerrvviieeww SSttaattuuttoo...,annual_report.pdf
4,5,Company Overview Statutory Reports Financial S...,Company Overview Statutory Reports Financial S...,annual_report.pdf


In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50, separators=["\n\n", "\n", ". ", " ", ""]
)

In [9]:
sample_text = """
Artificial Intelligence is transforming industries.
Machine learning enables systems to improve automatically.
Deep learning uses neural networks.
Retrieval-Augmented Generation combines retrieval and generation.
"""

chunks = text_splitter.split_text(sample_text)

for idx, chunk in enumerate(chunks, start=1):

    print(f"\nChunk {idx}\n")

    print(chunk)


Chunk 1

Artificial Intelligence is transforming industries.
Machine learning enables systems to improve automatically.
Deep learning uses neural networks.
Retrieval-Augmented Generation combines retrieval and generation.


In [10]:
first_page = pages_df.loc[0, "clean_text"]

chunks = text_splitter.split_text(first_page)

print("Number of Chunks:", len(chunks))

Number of Chunks: 1


In [11]:
for idx, chunk in enumerate(chunks, start=1):

    print("=" * 60)

    print(f"Chunk {idx}")

    print("=" * 60)

    print(chunk)

Chunk 1
ABBOTT INDIA LIMITED ANNUAL REPORT 2024-25 TOGETHER TOWARDS WELLNESS


In [13]:
chunk_records = []

chunk_counter = 1

for _, row in pages_df.iterrows():

    text = str(row["clean_text"]).strip()

    # Skip empty pages
    if not text:
        continue

    page_chunks = text_splitter.split_text(text)

    for chunk in page_chunks:

        chunk_records.append(
            {
                "chunk_id": chunk_counter,
                "source_document": row["source_document"],
                "page_number": row["page_number"],
                "chunk_text": chunk,
            }
        )

        chunk_counter += 1

In [14]:
chunks_df = pd.DataFrame(chunk_records)

chunks_df.head()

,chunk_id,source_document,page_number,chunk_text
0,1,annual_report.pdf,1,ABBOTT INDIA LIMITED ANNUAL REPORT 2024-25 TOG...
1,2,annual_report.pdf,2,Abbott India Limited CONTENTS Company Overview...
2,3,annual_report.pdf,2,of Directors 14 Independent Auditor’s Report 1...
3,4,annual_report.pdf,2,34 get and stay healthy.” Munir Shaikh Wellnes...
4,5,annual_report.pdf,2,. Wellness for Environment 54 We cannot guaran...


In [15]:
print("Total Pages:", len(pages_df))

print("Total Chunks:", len(chunks_df))

Total Pages: 125
Total Chunks: 1804


In [16]:
chunks_df["chunk_length"] = chunks_df["chunk_text"].apply(len)

chunks_df["chunk_length"].describe()

count    1804.000000
mean      376.320953
std       122.853928
min         3.000000
25%       313.000000
50%       413.500000
75%       480.000000
max       500.000000
Name: chunk_length, dtype: float64

In [17]:
OUTPUT_PATH = "data/processed/" "document_chunks.csv"

chunks_df.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)

Saved: data/processed/document_chunks.csv


In [18]:
def create_document_chunks(pages_dataframe, chunk_size=500, chunk_overlap=50):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunk_records = []

    chunk_counter = 1

    for _, row in pages_dataframe.iterrows():

        chunks = splitter.split_text(row["clean_text"])

        for chunk in chunks:

            chunk_records.append(
                {
                    "chunk_id": chunk_counter,
                    "source_document": row["source_document"],
                    "page_number": row["page_number"],
                    "chunk_text": chunk,
                }
            )

            chunk_counter += 1

    return pd.DataFrame(chunk_records)

In [20]:
# Replace NaN with empty strings
pages_df["clean_text"] = pages_df["clean_text"].fillna("").astype(str).str.strip()

# Remove empty pages
pages_df = pages_df[pages_df["clean_text"] != ""].reset_index(drop=True)

print(f"Pages after cleaning: {len(pages_df)}")

Pages after cleaning: 124


In [21]:
chunks_df = create_document_chunks(pages_df)

chunks_df.head()

,chunk_id,source_document,page_number,chunk_text
0,1,annual_report.pdf,1,ABBOTT INDIA LIMITED ANNUAL REPORT 2024-25 TOG...
1,2,annual_report.pdf,2,Abbott India Limited CONTENTS Company Overview...
2,3,annual_report.pdf,2,of Directors 14 Independent Auditor’s Report 1...
3,4,annual_report.pdf,2,34 get and stay healthy.” Munir Shaikh Wellnes...
4,5,annual_report.pdf,2,. Wellness for Environment 54 We cannot guaran...
